## Data Cleaning

Imports and loading the data

In [5]:
import pandas as pd 
import re

In [6]:
df_2023 = pd.read_csv(r"../CSV/Dataminers2023.csv")
df_2024 = pd.read_csv(r"../CSV/Dataminers2024.csv")
df_2025 = pd.read_csv(r"../CSV/Dataminers2025.csv")
df_2026 = pd.read_csv(r"../CSV/Dataminers2026.csv")

### Rename columns and merge the yearly files
The four survey files use different column headers. Each is renamed to a common set of names, then the files are concatenated. `source_file` records which file a row came from, while `response_year` is extracted from the timestamp itself. The `password` column is dropped.

In [7]:
canonical = ["timestamp", "password", "shoe_size", "programme",
             "height", "why", "seattle"]

renames = {
    2023: {
        "Timestamp": "timestamp",
        "Your mean shoe size (In European Continental system)  ": "shoe_size",
        "Which programme are you studying?": "programme",
        "Your height (in International inches)": "height",
        "Why are you taking this course?": "why",
        'How many letters are there in the word "Seattle"?': "seattle",
    },
    2025: {
        "Timestamp": "timestamp",
        "Enter_Your_Password": "password",
        "Your_mean_shoe_size__In_the_European_Continental_system_": "shoe_size",
        "Which_program_are_you_studying_": "programme",
        "Your_height__in_International_inches_": "height",
        "Why_are_you_taking_this_course_": "why",
        "How_many_letters_are_there_in_the_word__Seattle__": "seattle",
    },
}

renames[2024] = renames[2026] = {
    "Timestamp": "timestamp",
    "Enter Your Password": "password",
    "Your mean shoe size (In the European Continental system)": "shoe_size",
    "Which program are you studying?": "programme",
    "Your height (in International inches)": "height",
    "Why are you taking this course?": "why",
    'How many letters are there in the word "Seattle"?': "seattle",
}

frames = []
for year, df in [(2023, df_2023), (2024, df_2024), (2025, df_2025), (2026, df_2026)]:
    df = df.rename(columns=renames[year])
 
    df["source_file"] = year    
    frames.append(df)


combined = pd.concat(frames, ignore_index=True)
combined["response_year"] = combined["timestamp"].str.extract(r"(\d{4})").astype(int)
combined = combined[canonical + ["source_file", "response_year"]] 

combined = combined.drop(columns="password")

### Height
The question asked for height in inches, but respondents answered in many formats (cm, metres, feet and inches, decimal feet). `to_cm` parses these and classifies each value by magnitude. If a value can't be turned into a height, the function returns "None" and the cell is left empty.

In [8]:
FEET_RE = re.compile(r"^(\d+)\s*['\u2019]+\s*(\d+)?")
 
def to_cm(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
 

    m = FEET_RE.match(s)
    if m:
        feet = int(m.group(1))
        inches = int(m.group(2) or 0)
        if feet > 8:        
            return None      
        return (feet * 12 + inches) * 2.54
 
    explicit_cm = "cm" in s
    s = re.sub(r"(cm|centimeters?|centimetres?|inches|inch|in|m)\b", "", s).strip()
    s = s.replace(",", ".")
 
    try:
        x = float(s)
    except ValueError:
        return None
 
    if explicit_cm or x > 100:
        return x                    
    if x < 2:
        return x * 100               
    if x < 10:
        return x * 30.48              
    return x * 2.54                   
 
combined["height_cm"] = combined["height"].map(to_cm).round(1)

### Shoe size 
Unit labels are stripped and the number is extracted. Values below 20 are assumed to be US sizes and converted approximately to EU. Implausibly large values are treated as missing.

In [9]:
def shoe_eu(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r"(eu|euro|european|us|uk|size)", "", s).strip()
    s = s.replace(",", ".")
    s = re.sub(r"^[^\d]+", "", s) 
    try:
        x = float(s)
    except ValueError:
        return None
    if x < 20:          
        return round(x + 33, 1)
    if x > 60:                 
        return None
    return x

combined["shoe_eu"] = combined["shoe_size"].map(shoe_eu)
combined["shoe_eu"] = pd.to_numeric(combined["shoe_eu"])

### Programme 
The free text answers are mapped to a small set of standard programme labels.

In [10]:
def tidy_programme(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower().replace("digitial", "digital")
    if "software design" in s:      return "Software Design"
    if "digital innovation" in s:   return "Digital Innovation and Management"
    if "digital design" in s:       return "Digital Design"
    if "games" in s:                return "Games"
    if "computer science" in s:     return "Computer Science"
    if "elsewhere" in s:            return "Elsewhere (non-ITU)"
    return "Other"
 
combined["programme_clean"] = combined["programme"].map(tidy_programme)


### Remove the junk row
One from 2025 consisted of joke answers (e.g. height given as "1 ton") so we decided to remove it. 

In [11]:
junk = combined["height"].astype(str).str.strip().str.lower() == "1 ton"
"dropping", junk.sum(), "row(s):"
combined.loc[junk, ["height", "shoe_size", "seattle", "why"]].to_string()

combined = combined[~junk].reset_index(drop=True)
assert len(combined) == 222, len(combined)

### Impute missing heights
Missing heights are filled with the median. `height_imputed` keeps track of which rows were filled.

In [12]:
combined["height_imputed"] = combined["height_cm"].isna()
combined["height_cm"] = combined["height_cm"].fillna(
    round(combined["height_cm"].median(), 1)
)

### Motivation ("Why are you taking this course?")
The free-text answers are normalised (lower case, trimmed, single spaces) and turned into the binary features: 
* Interest
* Mandatory
* Commercial_skills
* Uncertain
* Less_attractive

In [13]:
combined["why_normalized"] = (
    combined["why"]
    .fillna("")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

motivation_options = {
    "exciting": [
        "it is the most exciting",
        "it seemed interesting",
        "Machinelearning is awesome"
        
    ],
    "mandatory": [
        "this was mandatory for me",
        "Specialisation",
        "It is a part of my specialization, and I felt good about picking a ready-made specialization direction"
    ],

    "commercial_skills": [
        "it'll give me commercially useful skills",
        "it'll give me business skills",
        "it'll give me thesis relevant skills",
        "I am interested in natural science & engineering and data mining is a good tool for these fields",
        "I wish to become the Data Mining Guru at my current workplace",
        "to solve/optimise issues in the construction industry",
    ],
    "uncertain": [
        "i'm not sure, let's see how it works out",
    ], 
    "less_attractive": [
        "The other courses were even less attractive",
    ],
    "garbage": [
        "My favorite ice cream is cheese pie",
        "Data Attraction",
        "Leon asked me to help out"
    ]
}


for feature, phrases in motivation_options.items():
    combined[feature] = (
        combined["why_normalized"]
        .apply(
            lambda answer: int(
                any(phrase.lower() in answer for phrase in phrases)
            )
        )
    )

### Timestamps 
Timestamps are parsed and converted to Copenhagen local time, and used to derive time-of-day, days since the cohort opened, and the free-choice cohort column.

In [14]:
ts = (combined["timestamp"].astype("string").str.strip()
        .str.replace(r"\s+(GMT[+-]\d{1,2}|[A-Z]{2,5})$", "", regex=True))

DAYFIRST = {2023: True, 2024: False, 2025: False, 2026: False}

parsed = pd.Series(index=combined.index, dtype="object")
for src, dayfirst in DAYFIRST.items():
    m = combined["source_file"] == src
    parsed[m] = ts[m].apply(
        lambda x: pd.to_datetime(x, format="mixed", dayfirst=dayfirst, errors="coerce"))

parsed = parsed.apply(lambda x: x.replace(tzinfo=None) if pd.notna(x) and x.tzinfo else x)

combined["timestamp"] = pd.to_datetime(parsed, errors="coerce")
combined["timestamp_clean"] = combined["timestamp"].dt.strftime("%d/%m/%Y %H:%M")


SHIFT = {2023: 0, 2024: -1, 2025: -7, 2026: 9}

combined["timestamp_cph"] = combined["timestamp"] + combined["source_file"].map(
    lambda y: pd.Timedelta(hours=SHIFT[y])
)

combined["hour"] = combined["timestamp_cph"].dt.hour 

combined["time_of_day"] = pd.cut(
    combined["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"],
)

combined["days_after_open"] = (
    combined["timestamp_cph"]
    - combined.groupby("response_year")["timestamp_cph"].transform("min")
).dt.days


combined["free_choice"] = (combined["response_year"] == 2026).astype(int)

### Export data
Keep the columns needed and save the cleaned data as a CSV file

In [15]:
final_cols = [
    "source_file", "response_year",
    "timestamp_cph",
    "time_of_day", "days_after_open",
    "height_cm",
    "shoe_eu",
    "programme_clean",
    "seattle",
    "why", "exciting", "mandatory", "commercial_skills", "uncertain", "less_attractive"
]
clean = combined[final_cols]
clean.to_csv("dataminers_cleaner_v2.csv", index=False)